# HDBSCAN — Manhattan


In [ ]:
import time
import tracemalloc
from pathlib import Path
import hdbscan
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, pairwise_distances
from sklearn.preprocessing import StandardScaler, Normalizer

# Path portabel: saat membuka notebook langsung, gunakan folder berisi cafes.csv
# sebagai working directory kernel, atau ubah path default di bawah ini.
# Runner summarize_notebooks.py memasok kedua path ini secara otomatis.
DATA_PATH = Path(globals().get("DATA_PATH", "cafes.csv")).expanduser().resolve()
FIGURES_DIR = Path(globals().get(
    "FIGURES_DIR", DATA_PATH.parent / "reports" / "figures"
)).expanduser().resolve()
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"CSV tidak ditemukan: {DATA_PATH}. Letakkan cafes.csv di folder kerja "
        "kernel atau ubah DATA_PATH pada sel ini."
    )
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# ==============================================================================
# BAGIAN A: LOAD DATA & PREPROCESSING (HDBSCAN)
# ==============================================================================
print("\n" + "="*56)
print("  BAGIAN A: LOAD DATA & PREPROCESSING (HDBSCAN)")
print("="*56)

df = pd.read_csv(DATA_PATH, on_bad_lines="skip")
kolom_fitur = ['latitude', 'longitude', 'rating', 'review_count']
kolom_ada = [k for k in kolom_fitur if k in df.columns]
df_fitur = df[kolom_ada].dropna()

# Standardisasi fitur numerik
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_fitur)

print(f"Data siap | Shape: {X_scaled.shape} | Baris: {len(df_fitur)}")


In [ ]:
MCS_MANH = 4
MS_MANH = 2

# --- 2. HDBSCAN MANHATTAN ---
print("\n" + "="*56)
print("  HDBSCAN — METRIK: MANHATTAN")
print(f"  Parameter: min_cluster_size={MCS_MANH}, min_samples={MS_MANH}")
print("="*56)

tracemalloc.start()
t_mulai = time.time()

hdb_manhattan = hdbscan.HDBSCAN(
    min_cluster_size=MCS_MANH, min_samples=MS_MANH, metric='manhattan', gen_min_span_tree=True
)
lbl_manhattan = hdb_manhattan.fit_predict(X_scaled)

t_selesai = time.time()
_, mem_puncak = tracemalloc.get_traced_memory()
tracemalloc.stop()

n_klaster_m = len(set(lbl_manhattan)) - (1 if -1 in lbl_manhattan else 0)
n_noise_m = int((lbl_manhattan == -1).sum())
pct_noise_m = n_noise_m / len(lbl_manhattan) * 100
mask_valid_m = lbl_manhattan != -1

if mask_valid_m.sum() > 1 and n_klaster_m >= 2:
    sil_manhattan = silhouette_score(X_scaled[mask_valid_m], lbl_manhattan[mask_valid_m], metric='manhattan')
    sil_m_str = f"{sil_manhattan:.4f}"
else:
    sil_manhattan = None
    sil_m_str = "N/A (klaster < 2)"

waktu_m = t_selesai - t_mulai
mem_m_kb = mem_puncak / 1024
mem_m_mb = mem_puncak / (1024**2)

print(f"  Jumlah Klaster    : {n_klaster_m}")
print(f"  Titik Noise (-1)  : {n_noise_m} ({pct_noise_m:.1f}%)")
print(f"  Waktu Eksekusi    : {waktu_m:.4f} detik")
print(f"  Memori Puncak     : {mem_m_kb:.2f} KB ({mem_m_mb:.4f} MB)")
print(f"  Silhouette Score  : {sil_m_str}")
print("="*56)


In [ ]:
def plot_condensed_tree_safe(model, ax):
    """Plot tree dengan label/ellipse skalar, kompatibel dengan NumPy 2.x."""
    from matplotlib.patches import Ellipse
    from matplotlib.text import Annotation

    model.condensed_tree_.plot(axis=ax, select_clusters=True, label_clusters=True)
    # HDBSCAN memakai np.diff untuk tinggi minimum: hasilnya array satu elemen.
    # Konversi eksplisit sebelum tight_layout/savefig/show memicu proses render.
    for label in ax.texts:
        if isinstance(label, Annotation):
            label.xy = tuple(float(np.asarray(v).item()) for v in label.xy)
            label.set_position(tuple(float(np.asarray(v).item())
                                     for v in label.get_position()))
    for patch in ax.patches:
        if isinstance(patch, Ellipse):
            patch.set_width(float(np.asarray(patch.width).item()))
            patch.set_height(float(np.asarray(patch.height).item()))
    return ax


# ==============================================================================
# BAGIAN D: VISUALISASI PCA 2D & CONDENSED CLUSTER TREE
# ==============================================================================
print("\n" + "="*56)
print("  BAGIAN D: VISUALISASI PCA 2D (HDBSCAN)")
print("="*56)

pca = PCA(n_components=2, random_state=42)
X_pca_std = pca.fit_transform(X_scaled)
v1_std, v2_std = pca.explained_variance_ratio_ * 100

fig, ax = plt.subplots(figsize=(6, 5.5), dpi=150)
axes = [ax]

def plot_hdbscan_pca(ax, X_pca_coords, lbls, v1, v2, title):
    mask_noise = lbls == -1
    mask_klaster = ~mask_noise

    # Plot titik noise -> hitam, tanda x
    ax.scatter(X_pca_coords[mask_noise, 0], X_pca_coords[mask_noise, 1],
               c='black', marker='x', s=35, alpha=0.7,
               label=f'Noise ({mask_noise.sum()} titik)')

    # Plot titik klaster -> berwarna
    if mask_klaster.sum() > 0:
        sc = ax.scatter(X_pca_coords[mask_klaster, 0], X_pca_coords[mask_klaster, 1],
                        c=lbls[mask_klaster], cmap='Set1', s=30, alpha=0.8,
                        label='Klaster')
        plt.colorbar(sc, ax=ax, label='Klaster')

    ax.set_title(title, fontweight='bold')
    ax.set_xlabel(f'PCA 1 ({v1:.1f}%)')
    ax.set_ylabel(f'PCA 2 ({v2:.1f}%)')
    ax.legend(fontsize=9)
    ax.grid(True, linestyle='--', alpha=0.4)

plot_hdbscan_pca(axes[0], X_pca_std, lbl_manhattan, v1_std, v2_std,
                 f"HDBSCAN Manhattan\n{n_klaster_m} klaster, {n_noise_m} noise | Sil: {sil_m_str}")

plt.suptitle("Visualisasi PCA 2D HDBSCAN (Data Numerik Kafe)", fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
hdb_pca_path = FIGURES_DIR / "hdbscan_manhattan_pca_plots.png"
plt.savefig(hdb_pca_path, dpi=150, bbox_inches='tight')
plt.show()
plt.close()
print(f"Visualisasi PCA disimpan ke: {hdb_pca_path}")

fig, ax = plt.subplots(figsize=(7.5, 5.5), dpi=150)
axes = [ax]
# Condensed Tree Plot
plot_condensed_tree_safe(hdb_manhattan, axes[0])
axes[0].set_title(f"Condensed Tree (Manhattan, mcs={MCS_MANH})", fontweight='bold')
axes[0].set_ylabel(r"$\lambda = 1 / \text{distance}$")

plt.tight_layout()
tree_plot_path = FIGURES_DIR / "hdbscan_manhattan_condensed_tree.png"
plt.savefig(tree_plot_path, dpi=150)
plt.show()
plt.close()
print(f"Condensed Tree plot disimpan ke: {tree_plot_path}")


In [ ]:
# Dictionary ringkasan performa untuk pelaporan
HDBSCAN_SUMMARY = {
    "manhattan": {
        "algo": "HDBSCAN", "metric": "Manhattan", "param": f"mcs={MCS_MANH}, ms={MS_MANH}",
        "time": waktu_m, "mem_mb": mem_m_mb, "clusters": n_klaster_m,
        "noise_pct": pct_noise_m, "silhouette": sil_manhattan, "labels": lbl_manhattan
    }
}
